# FP seq2 Perturb-seq cNMF program discovery

This cleaned notebook documents the cNMF workflow used to derive the endothelial gene programs used downstream in the FP seq2 Perturb-seq analysis.

- Inputs and outputs use descriptive relative paths.
- Absolute project-specific directories have been removed for release.
- The archived exploratory notebook records an earlier `k = 80` endpoint; this release notebook documents the retained production workflow used for downstream 100-program analyses.
- Program annotation and SCEPTRE-based perturbation testing are handled in separate release scripts.

In [ ]:
from pathlib import Path

import numpy as np
import scanpy as sc
from cnmf import cNMF

np.random.seed(14)

## Define descriptive input and output locations

Replace the example paths below with the deposited endothelial AnnData object and the output directory you want to use for cNMF products.

In [ ]:
input_h5ad_path = Path("input/perturbseq_endothelial_cells_with_raw_counts.h5ad")
output_root = Path("output/fp_seq2_cnmf")

exploratory_run_label = "NoBatchCorrection"
production_run_label = "NoBatchCorrection_k100"

exploratory_components = [15]
rank_sweep_components = [10, 20, 30, 35, 40, 45, 50, 60]
production_components = [100]

exploratory_iterations = 20
rank_sweep_iterations = 20
production_iterations = 200

exploratory_highvar_genes = 2000
production_highvar_genes = 10000

exploratory_density_threshold = 0.1
production_density_threshold = 0.2

num_workers = 12
output_root.mkdir(parents=True, exist_ok=True)

## Load the AnnData object and restore raw counts

The original analysis restored raw UMI counts from the `raw` slot before cNMF so that factorization was run on count-scale expression values rather than on a transformed matrix.

In [ ]:
adata = sc.read_h5ad(input_h5ad_path)
adata.X = adata.raw.X.copy()

if "_index" in adata.raw.var.columns:
    adata.raw.var = adata.raw.var.rename(columns={"_index": "features"})

adata

## Exploratory cNMF run at `k = 15`

The archived notebook first used a small consensus run to visualize broad structure and to confirm that batch-associated signal remained without correction.

In [ ]:
initial_cnmf = cNMF(output_dir=str(output_root), name=exploratory_run_label)
initial_cnmf.prepare(
    counts_fn=str(input_h5ad_path),
    components=exploratory_components,
    n_iter=exploratory_iterations,
    seed=14,
    num_highvar_genes=exploratory_highvar_genes,
)
initial_cnmf.factorize(worker_i=0, total_workers=1)
initial_cnmf.combine()
initial_cnmf.consensus(
    k=15,
    density_threshold=exploratory_density_threshold,
    show_clustering=True,
    close_clustergram_fig=False,
)

initial_usage, initial_spectra_scores, initial_spectra_tpm, initial_top_genes = initial_cnmf.load_results(
    K=15,
    density_threshold=exploratory_density_threshold,
)
initial_usage.head()

## Rank sweep across candidate program numbers

The archived notebook next prepared a rank sweep across multiple candidate values of `k` and used the command-line `k_selection_plot` utility to inspect the tradeoff between stability and reconstruction.

In [ ]:
rank_sweep_cnmf = cNMF(output_dir=str(output_root), name=exploratory_run_label)
rank_sweep_cnmf.prepare(
    counts_fn=str(input_h5ad_path),
    components=rank_sweep_components,
    n_iter=rank_sweep_iterations,
    seed=123,
    num_highvar_genes=exploratory_highvar_genes,
)

rank_sweep_factorize_command = (
    f"parallel cnmf factorize --output-dir {output_root.as_posix()} "
    f"--name {exploratory_run_label} --total-worker {num_workers} --worker-index {{}} ::: "
    + " ".join(map(str, range(num_workers)))
)
rank_sweep_combine_command = (
    f"cnmf combine --output-dir {output_root.as_posix()} --name {exploratory_run_label}"
)
rank_sweep_k_selection_command = (
    f"cnmf k_selection_plot --output-dir {output_root.as_posix()} --name {exploratory_run_label}"
)

print(rank_sweep_factorize_command)
print(rank_sweep_combine_command)
print(rank_sweep_k_selection_command)

## Final production run at `k = 100`

The retained downstream program-testing workflow uses a later production rerun with 100 consensus programs and a local-density threshold of 0.2. The larger run follows the same general structure as the archived notebook but uses a larger highly variable gene set and 200 factorization replicates. The original notebook does not explicitly record the seed for this later rerun; the `seed = 123` setting below mirrors the larger archived run and should be updated if exact provenance requires a different value.

In [ ]:
production_cnmf = cNMF(output_dir=str(output_root), name=production_run_label)
production_cnmf.prepare(
    counts_fn=str(input_h5ad_path),
    components=production_components,
    n_iter=production_iterations,
    seed=123,
    num_highvar_genes=production_highvar_genes,
)

production_factorize_command = (
    f"parallel cnmf factorize --output-dir {output_root.as_posix()} "
    f"--name {production_run_label} --skip-completed-runs --total-worker {num_workers} --worker-index {{}} ::: "
    + " ".join(map(str, range(num_workers)))
)
production_combine_command = (
    f"cnmf combine --output-dir {output_root.as_posix()} --name {production_run_label}"
)
production_consensus_command = (
    f"cnmf consensus --output-dir {output_root.as_posix()} --name {production_run_label} "
    "--components 100 --local-density-threshold 0.2 --show-clustering"
)

print(production_factorize_command)
print(production_combine_command)
print(production_consensus_command)

## Load the retained consensus outputs

These objects are the main release products that feed downstream annotation, subtype enrichment, and program-level perturbation testing.

In [ ]:
production_usage, production_spectra_scores, production_spectra_tpm, production_top_genes = production_cnmf.load_results(
    K=100,
    density_threshold=production_density_threshold,
)
production_usage.head()